# Kimi K3 — Production Invoice Intelligence API

## NVIDIA NIM + Accounting Knowledge Base + Fixed JSON Structure + Runtime Zoho COA + FastAPI + ngrok

**Production-only notebook.**

- NVIDIA NIM is retained.
- The accounting Knowledge Base is loaded once per Colab runtime.
- The API output uses the exact fixed JSON structure supplied for this project.
- No JSON Schema lock.
- No `additionalProperties=false`.
- No `Draft202012Validator`.
- No schema audit.
- No schema-repair loop.
- Backend remains the final authority for accounting/tax calculations and statutory validation.
- The FastAPI server runs in the foreground so requests and extracted JSON appear live in the Colab output.

### API

- `GET /health`
- `POST /api/infer/k3-invoice`
- `POST /api/infer/extract-invoice` (compatibility alias)

### Runtime context

The backend sends:
1. One invoice PDF/image as base64.
2. The complete active Zoho COA in `chart_of_accounts`.

K3 uses that COA only for account matching and never invents Zoho account IDs.


In [ ]:
# 1) Install dependencies and imports
!pip -q install -U requests pymupdf pillow fastapi uvicorn nest_asyncio pyngrok python-multipart

import base64
import io
import json
import os
import re
import threading
import time
import traceback
import datetime
from pathlib import Path
from getpass import getpass
from typing import Optional, List, Dict, Any

import requests
import fitz
from PIL import Image

import nest_asyncio
import uvicorn
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field
from pyngrok import ngrok

nest_asyncio.apply()

print("Dependencies ready.")


In [ ]:
# 2) NVIDIA NIM runtime configuration + one-time Knowledge Base upload

MODEL = "moonshotai/kimi-k3"
BASE_URL = "https://integrate.api.nvidia.com/v1/chat/completions"

TEMPERATURE = 0.0
REASONING_EFFORT = "low"
MAX_TOKENS = 8192

CONNECT_TIMEOUT = 30
READ_TIMEOUT = 360

# Keep retries bounded. Backend/demo should never spam NIM.
MAX_HTTP_RETRIES = 2

RETRY_BACKOFF_SECONDS = 15
MAX_RETRY_WAIT_SECONDS = 180
MIN_SECONDS_BETWEEN_REQUESTS = 8

K3_PORT = 8000

# The Windows project path is not available inside Colab.
# Upload the exact project KB once to this Colab runtime.
KB_PATH = Path("/content/KIMI_K3_ACCOUNTING_SPEC_V2.md")

if not KB_PATH.exists():
    from google.colab import files

    print("Please upload the exact project Knowledge Base:")
    print("KIMI_K3_ACCOUNTING_SPEC_V2.md")

    uploaded = files.upload()

    if "KIMI_K3_ACCOUNTING_SPEC_V2.md" in uploaded:
        KB_PATH.write_bytes(
            uploaded["KIMI_K3_ACCOUNTING_SPEC_V2.md"]
        )
    elif uploaded:
        first_name = next(iter(uploaded))
        KB_PATH.write_bytes(uploaded[first_name])
    else:
        raise FileNotFoundError(
            "Knowledge Base upload cancelled."
        )

if not KB_PATH.exists():
    raise FileNotFoundError(
        f"Knowledge Base not found: {KB_PATH}"
    )

KNOWLEDGE_BASE_TEXT = KB_PATH.read_text(
    encoding="utf-8"
)

print("=" * 90)
print("NVIDIA KIMI K3 CONFIGURATION")
print("=" * 90)
print("Model        :", MODEL)
print("Endpoint     :", BASE_URL)
print("Max tokens   :", MAX_TOKENS)
print("KB           :", KB_PATH.name)
print("KB chars     :", len(KNOWLEDGE_BASE_TEXT))
print("=" * 90)


In [ ]:
# 3) EXACT FIXED JSON STRUCTURE
#
# This is the project output shape.
# It is NOT a JSON Schema validator.
# Do not add/remove/rename these structural keys in the prompt.
#
# Array element structures are intentionally left open because
# the project only fixed the container structure for those arrays.

FIXED_JSON_STRUCTURE = {
    "invoice_details": {
        "invoice_number": None,
        "invoice_date": None,
        "due_date": None,
        "po_number": None,
        "place_of_supply": None,
        "payment_terms": None,
        "currency": None,
        "document_type": None,
    },

    "vendor_details": {
        "vendor_name": None,
        "vendor_address": None,
        "vendor_gstin": None,
        "vendor_pan": None,
        "bank_details": None,
    },

    "customer_details": {
        "customer_name": None,
        "customer_address": None,
        "customer_gstin": None,
        "customer_pan": None,
    },

    "line_items": [
        {
            "line_index": None,
            "description": None,
            "quantity": None,
            "unit": None,
            "unit_price": None,
            "discount": None,
            "taxable_amount": None,
            "hsn_sac": None,
            "gst_rate": None,
            "cgst_amount": None,
            "sgst_amount": None,
            "igst_amount": None,
        }
    ],

    "financial_details": {
        "subtotal": None,
        "discount_total": None,
        "taxable_amount": None,
        "tax_total": None,
        "cgst_amount": None,
        "sgst_amount": None,
        "igst_amount": None,
        "round_off": None,
        "total_amount": None,
    },

    "gst_support": {
        "supply_type_candidate": None,
        "tax_components_candidate": None,
        "rcm_candidate": None,
        "gst_rate_candidates": [],
        "rate_requires_external_validation": False,
        "reason": None,
    },

    "tds_support": {
        "tds_applicable_candidate": None,
        "payment_nature": None,
        "law_version_candidate": None,
        "provision_candidate": None,
        "legacy_provision_reference": None,
        "rate_candidate": None,
        "base_candidate": None,
        "threshold_status": None,
        "pan_status": None,
        "cumulative_vendor_data_required": False,
        "requires_backend_validation": True,
        "reason": None,
    },

    "tcs_support": {
        "tcs_applicable_candidate": None,
        "provision_candidate": None,
        "requires_review": False,
        "reason": None,
    },

    "itc_support": {
        "candidate": None,
        "business_use": None,
        "document_sufficiency": None,
        "gstr2b_status": None,
        "blocked_credit_risk": None,
        "apportionment_risk": None,
        "payment_180_day_risk": None,
        "time_limit_status": None,
        "eligible_amount_candidate": None,
        "requires_backend_validation": True,
        "reason": None,
    },

    "coa_support": {
        "line_matches": [],
    },

    "gl_support": {
        "line_classifications": [],
        "journal_pattern": None,
        "requires_backend_generation": True,
    },

    "validation": {
        "subtotal_mismatch": False,
        "tax_mismatch": False,
        "total_mismatch": False,
        "discount_ambiguity": False,
        "round_off_issue": False,
    },

    "review_flags": [],
}

FIXED_JSON_EXAMPLE = json.dumps(
    FIXED_JSON_STRUCTURE,
    indent=2,
    ensure_ascii=False,
)

print("Fixed JSON structure loaded.")
print(FIXED_JSON_EXAMPLE)


In [ ]:
# 4) Fixed-structure prompt for Kimi K3

GOD_LEVEL_K3_SYSTEM_PROMPT = r'''
You are Kimi K3, the production invoice-intelligence and accounting
decision-support layer for an Indian Accounts Payable system.

Your job:
1. Read the complete invoice document faithfully.
2. Extract observable invoice facts.
3. Interpret/classify the transaction using the supplied accounting knowledge base.
4. Use the complete runtime Zoho COA supplied in the user request.
5. Provide accounting/tax decision-support candidates.
6. Flag uncertainty, contradictions, missing information and anomalies.
7. Never fabricate facts.

The backend is the final authority for:
- GST calculations
- TDS calculations
- ITC calculations
- statutory validation
- journal generation
- persistence
- Zoho export

========================================================
FIXED OUTPUT STRUCTURE — ABSOLUTE
========================================================

Return exactly ONE JSON object using the EXACT structure supplied
in the user message.

Use exactly these top-level keys:

invoice_details
vendor_details
customer_details
line_items
financial_details
gst_support
tds_support
tcs_support
itc_support
coa_support
gl_support
validation
review_flags

Do not:
- add top-level fields
- remove top-level fields
- rename fields
- invent aliases
- return Markdown
- return code fences
- return commentary outside JSON
- return chain-of-thought

The fixed structure shown by the user is the API structure.
The Knowledge Base may contain older/example JSON structures.
Those examples MUST NOT replace or alter the fixed API structure.

For any field where the invoice does not provide evidence, use null
when the fixed structure uses null.

Explicit zero is different from missing.
Do not convert missing values into zero.

========================================================
INVOICE EXTRACTION
========================================================

Extract invoice number, dates, PO number, place of supply,
payment terms, currency and document type into invoice_details.

Extract supplier details into vendor_details.
Extract buyer details into customer_details.

Never invent:
- invoice number
- invoice date
- GSTIN
- PAN
- bank details
- quantity
- prices
- discounts
- tax values
- totals

If vendor PAN is not printed but a valid vendor GSTIN is visible,
the project rule may be used to derive the vendor PAN.
Never derive vendor PAN from customer GSTIN.

========================================================
LINE ITEMS
========================================================

Return one line_items record per material invoice line.

Use exactly these line-level fields:
line_index
description
quantity
unit
unit_price
discount
taxable_amount
hsn_sac
gst_rate
cgst_amount
sgst_amount
igst_amount

Do not invent a line item.
Do not double-count repeated headers or repeated rows.

========================================================
FINANCIAL DETAILS
========================================================

Use exactly:
subtotal
discount_total
taxable_amount
tax_total
cgst_amount
sgst_amount
igst_amount
round_off
total_amount

Extract invoice evidence faithfully.
Do not reconstruct missing values from arithmetic.

========================================================
GST SUPPORT
========================================================

Use exactly:
supply_type_candidate
tax_components_candidate
rcm_candidate
gst_rate_candidates
rate_requires_external_validation
reason

This is support/candidate information only.
Do not claim final statutory certainty.

========================================================
TDS SUPPORT
========================================================

Use exactly:
tds_applicable_candidate
payment_nature
law_version_candidate
provision_candidate
legacy_provision_reference
rate_candidate
base_candidate
threshold_status
pan_status
cumulative_vendor_data_required
requires_backend_validation
reason

Do not invent cumulative vendor history, threshold status, payer category,
residency, certificate facts or final statutory amounts.

When required evidence is unavailable, use null and flag backend review
through the fixed fields.

========================================================
TCS / ITC
========================================================

Keep TCS separate from TDS.

ITC is a candidate/support signal only.
Do not invent GSTR-2B status or final eligible amount.

========================================================
ZOHO COA
========================================================

The backend supplies the COMPLETE active Zoho COA in the request.

Use ONLY that supplied account universe.
Never fabricate a Zoho account ID.
Never create an account.
Never select an account from memory.

For each material invoice line, provide one corresponding
coa_support.line_matches record.

========================================================
GL SUPPORT
========================================================

Use:
line_classifications
journal_pattern
requires_backend_generation

GL generation itself is performed by the backend.

========================================================
VALIDATION
========================================================

Use:
subtotal_mismatch
tax_mismatch
total_mismatch
discount_ambiguity
round_off_issue

These are observed/interpretive indicators only.
The backend remains final authority.

========================================================
DOCUMENT ROBUSTNESS
========================================================

Use visual layout, table relationships, labels, typography, page order,
cross-page continuity and document structure.

Handle:
- digital invoices
- scanned invoices
- photographed invoices
- skewed/perspective documents
- handwritten/mixed content
- dense tables
- multi-page invoices
- color-heavy branded invoices
- ERP/Excel-like exports

========================================================
FINAL
========================================================

Return only the JSON object.
Follow the fixed structure exactly.
Do not copy older sample schemas from the knowledge base.
'''

print("K3 fixed-structure prompt loaded.")


In [ ]:
# 5) NVIDIA NIM credentials

NVIDIA_API_KEY = None

try:
    from google.colab import userdata
    NVIDIA_API_KEY = userdata.get("NVIDIA_API_KEY")
except Exception:
    pass

if not NVIDIA_API_KEY:
    NVIDIA_API_KEY = getpass(
        "Enter NVIDIA NIM API key: "
    ).strip()

if not NVIDIA_API_KEY:
    raise ValueError(
        "NVIDIA_API_KEY is empty."
    )

HEADERS = {
    "Authorization": f"Bearer {NVIDIA_API_KEY}",
    "Content-Type": "application/json",
    "Accept": "application/json",
}

print("=" * 90)
print("NVIDIA NIM AUTHENTICATION READY")
print("=" * 90)
print("Model:", MODEL)
print("Endpoint:", BASE_URL)
print("=" * 90)


In [ ]:
# 6) PDF / image preparation

def render_pdf_pages_bytes(
    raw_bytes: bytes,
    dpi: int = 150,
):
    doc = fitz.open(
        stream=raw_bytes,
        filetype="pdf",
    )

    pages = []
    matrix = fitz.Matrix(
        dpi / 72,
        dpi / 72,
    )

    try:
        for page_number, page in enumerate(doc):
            pix = page.get_pixmap(
                matrix=matrix,
                alpha=False,
            )

            pages.append({
                "page": page_number + 1,
                "bytes": pix.tobytes("jpeg"),
                "mime": "image/jpeg",
            })

    finally:
        doc.close()

    return pages


def compress_image_bytes(
    img_bytes: bytes,
    max_side: int = 1800,
    quality: int = 82,
):
    img = Image.open(
        io.BytesIO(img_bytes)
    ).convert("RGB")

    scale = min(
        1.0,
        max_side / max(
            img.width,
            img.height,
        ),
    )

    if scale < 1.0:
        img = img.resize(
            (
                max(
                    1,
                    int(img.width * scale),
                ),
                max(
                    1,
                    int(img.height * scale),
                ),
            ),
            Image.Resampling.LANCZOS,
        )

    out = io.BytesIO()

    img.save(
        out,
        format="JPEG",
        quality=quality,
        optimize=True,
    )

    return out.getvalue()


def to_data_uri(
    img_bytes: bytes,
    mime: str = "image/jpeg",
):
    return (
        f"data:{mime};base64,"
        f"{base64.b64encode(img_bytes).decode('utf-8')}"
    )


def decode_invoice_to_pages(
    raw_bytes: bytes,
    filename: str,
):
    suffix = Path(
        filename or ""
    ).suffix.lower()

    if (
        raw_bytes[:4] == b"%PDF"
        or suffix == ".pdf"
    ):

        pages = render_pdf_pages_bytes(
            raw_bytes
        )

        if not pages:
            raise ValueError(
                "PDF contains no pages."
            )

        return (
            "PDF",
            [
                {
                    "page": p["page"],
                    "bytes": compress_image_bytes(
                        p["bytes"]
                    ),
                    "mime": "image/jpeg",
                }
                for p in pages
            ],
        )

    try:
        img = Image.open(
            io.BytesIO(raw_bytes)
        ).convert("RGB")

    except Exception as exc:
        raise ValueError(
            f"Unsupported/invalid image: {exc}"
        )

    out = io.BytesIO()

    img.save(
        out,
        format="JPEG",
        quality=88,
        optimize=True,
    )

    return (
        "IMAGE",
        [
            {
                "page": 1,
                "bytes": compress_image_bytes(
                    out.getvalue()
                ),
                "mime": "image/jpeg",
            }
        ],
    )


print("Invoice PDF/image pipeline ready.")


In [ ]:
# 7) NVIDIA NIM client
#
# No JSON Schema validation.
# No schema lock.
# No schema-repair loop.
#
# The fixed structure is controlled by the prompt.
# Only JSON parsing + lightweight structural presence check are performed.

_last_request_finished_at = 0.0
NIM_REQUEST_LOCK = threading.Lock()


class K3RateLimitError(RuntimeError):
    pass


def _flush_print(*args, **kwargs):
    print(*args, flush=True, **kwargs)


def _throttle():
    global _last_request_finished_at

    elapsed = (
        time.time()
        - _last_request_finished_at
    )

    wait = (
        MIN_SECONDS_BETWEEN_REQUESTS
        - elapsed
    )

    if wait > 0:
        _flush_print(
            f"[NIM] Waiting {wait:.1f}s before next request..."
        )
        time.sleep(wait)


def _extract_json_object(text: str):
    if not text:
        raise ValueError(
            "Kimi K3 returned empty content."
        )

    cleaned = str(text).strip()

    cleaned = re.sub(
        r"<think>.*?</think>",
        "",
        cleaned,
        flags=re.DOTALL | re.IGNORECASE,
    ).strip()

    try:
        parsed = json.loads(cleaned)

        if isinstance(parsed, dict):
            return parsed

    except Exception:
        pass

    fenced = re.search(
        r"```(?:json)?\s*(\{.*?\})\s*```",
        cleaned,
        flags=re.DOTALL | re.IGNORECASE,
    )

    if fenced:
        parsed = json.loads(
            fenced.group(1)
        )

        if isinstance(parsed, dict):
            return parsed

    start = cleaned.find("{")
    end = cleaned.rfind("}")

    if (
        start >= 0
        and end > start
    ):
        parsed = json.loads(
            cleaned[
                start:end + 1
            ]
        )

        if isinstance(parsed, dict):
            return parsed

    raise ValueError(
        "Kimi K3 response did not contain a valid JSON object."
    )


def _normalize_zoho_coa(
    chart_of_accounts,
):
    normalized = []

    for account in (
        chart_of_accounts or []
    ):

        if not isinstance(
            account,
            dict,
        ):
            continue

        account_id = str(
            account.get("account_id")
            or account.get("id")
            or account.get("accountId")
            or account.get("zoho_account_id")
            or ""
        ).strip()

        account_name = str(
            account.get("account_name")
            or account.get("name")
            or account.get("accountName")
            or ""
        ).strip()

        account_type = str(
            account.get("account_type")
            or account.get("type")
            or account.get("accountType")
            or ""
        ).strip()

        description = str(
            account.get("description")
            or account.get("account_description")
            or ""
        ).strip()

        if not account_id or not account_name:
            continue

        normalized.append({
            "account_id": account_id,
            "account_name": account_name,
            "account_type": account_type,
            "description": description,
        })

    return normalized


def _build_user_content(
    invoice_filename,
    coa,
):
    return (
        f"INVOICE FILE: {invoice_filename}\n\n"
        "The attached page images are the complete invoice document.\n"
        "Return exactly ONE JSON object using the FIXED JSON STRUCTURE below.\n"
        "Do not add, remove, rename, or replace structural keys.\n"
        "Do not return Markdown or explanations.\n\n"
        "FIXED JSON STRUCTURE:\n"
        "---------------------\n"
        + FIXED_JSON_EXAMPLE
        + "\n---------------------\n\n"
        "RUNTIME ZOHO COA — COMPLETE BACKEND-SUPPLIED LIST\n"
        "Use only this account universe for COA matching.\n"
        "Never fabricate a Zoho account ID.\n\n"
        + json.dumps(
            coa,
            ensure_ascii=False,
            indent=2,
        )
    )


def _build_nim_payload(
    images,
    invoice_filename,
    coa,
):
    user_content = [
        {
            "type": "text",
            "text": _build_user_content(
                invoice_filename,
                coa,
            ),
        }
    ]

    for page in images:
        user_content.append({
            "type": "image_url",
            "image_url": {
                "url": to_data_uri(
                    page["bytes"],
                    page["mime"],
                )
            },
        })

    return {
        "model": MODEL,
        "messages": [
            {
                "role": "system",
                "content": (
                    GOD_LEVEL_K3_SYSTEM_PROMPT
                    + "\n\n================ ACCOUNTING KNOWLEDGE BASE ================\n"
                    + KNOWLEDGE_BASE_TEXT
                    + "\n================ END ACCOUNTING KNOWLEDGE BASE ================\n"
                ),
            },
            {
                "role": "user",
                "content": user_content,
            },
        ],
        "temperature": TEMPERATURE,
        "reasoning_effort": REASONING_EFFORT,
        "max_tokens": MAX_TOKENS,
        "stream": False,
    }


def _basic_fixed_structure_check(
    prediction,
):
    if not isinstance(
        prediction,
        dict,
    ):
        raise ValueError(
            "Kimi K3 output is not a JSON object."
        )

    expected_top_level = list(
        FIXED_JSON_STRUCTURE.keys()
    )

    missing = [
        key
        for key in expected_top_level
        if key not in prediction
    ]

    if missing:
        raise ValueError(
            "Kimi K3 returned JSON but the fixed "
            f"structure is incomplete. Missing top-level keys: {missing}"
        )

    return True


def _validate_coa_runtime_ids(
    prediction,
    coa,
):
    supplied_ids = {
        row["account_id"]
        for row in coa
    }

    coa_support = prediction.get(
        "coa_support"
    )

    if not isinstance(
        coa_support,
        dict,
    ):
        return

    matches = coa_support.get(
        "line_matches",
        [],
    )

    if not isinstance(
        matches,
        list,
    ):
        return

    for match in matches:

        if not isinstance(
            match,
            dict,
        ):
            continue

        account_id = (
            match.get(
                "matched_account_id"
            )
            or match.get(
                "account_id"
            )
        )

        if (
            account_id is not None
            and account_id
            not in supplied_ids
        ):
            raise ValueError(
                "Kimi K3 returned a Zoho account ID "
                f"not present in runtime COA: {account_id}"
            )


def _post_to_nim(
    payload,
):
    global _last_request_finished_at

    last_exception = None

    for attempt in range(
        1,
        MAX_HTTP_RETRIES + 1,
    ):

        _throttle()

        started = time.time()

        _flush_print(
            f"[NIM] Kimi K3 request "
            f"{attempt}/{MAX_HTTP_RETRIES}"
        )

        try:

            response = requests.post(
                BASE_URL,
                headers=HEADERS,
                json=payload,
                timeout=(
                    CONNECT_TIMEOUT,
                    READ_TIMEOUT,
                ),
            )

            _last_request_finished_at = (
                time.time()
            )

        except (
            requests.Timeout,
            requests.ConnectionError,
        ) as exc:

            _last_request_finished_at = (
                time.time()
            )

            last_exception = exc

            if (
                attempt
                < MAX_HTTP_RETRIES
            ):

                wait_s = min(
                    RETRY_BACKOFF_SECONDS
                    * (
                        2
                        ** (
                            attempt - 1
                        )
                    ),
                    MAX_RETRY_WAIT_SECONDS,
                )

                _flush_print(
                    f"[NIM] Connection/timeout error. "
                    f"Retrying after {wait_s:.1f}s"
                )

                time.sleep(
                    wait_s
                )

                continue

            raise TimeoutError(
                "Kimi K3 upstream "
                f"connection failed: {exc}"
            ) from exc


        status = response.status_code


        if status == 429:

            retry_after = (
                response.headers.get(
                    "Retry-After"
                )
            )

            if (
                attempt
                < MAX_HTTP_RETRIES
            ):

                try:
                    wait_s = (
                        float(retry_after)
                        if retry_after
                        else RETRY_BACKOFF_SECONDS
                        * (
                            2
                            ** (
                                attempt - 1
                            )
                        )
                    )
                except Exception:
                    wait_s = (
                        RETRY_BACKOFF_SECONDS
                        * (
                            2
                            ** (
                                attempt - 1
                            )
                        )
                    )

                wait_s = min(
                    max(
                        1.0,
                        wait_s,
                    ),
                    MAX_RETRY_WAIT_SECONDS,
                )

                _flush_print(
                    f"[NIM] HTTP 429. "
                    f"Retrying after {wait_s:.1f}s."
                )

                time.sleep(
                    wait_s
                )

                continue

            raise K3RateLimitError(
                "NVIDIA NIM rate limit/capacity reached "
                "(HTTP 429). "
                f"Retry-After="
                f"{retry_after or 'not provided'}"
            )


        if status in {
            500,
            502,
            503,
            504,
        }:

            last_exception = RuntimeError(
                f"NVIDIA NIM transient HTTP "
                f"{status}: {response.text[:1000]}"
            )

            if (
                attempt
                < MAX_HTTP_RETRIES
            ):

                wait_s = min(
                    RETRY_BACKOFF_SECONDS
                    * (
                        2
                        ** (
                            attempt - 1
                        )
                    ),
                    MAX_RETRY_WAIT_SECONDS,
                )

                _flush_print(
                    f"[NIM] HTTP {status}. "
                    f"Retrying after {wait_s:.1f}s."
                )

                time.sleep(
                    wait_s
                )

                continue

            raise last_exception


        if not response.ok:

            raise RuntimeError(
                f"NVIDIA NIM request failed "
                f"[{status}]: "
                f"{response.text[:1500]}"
            )


        elapsed = (
            time.time()
            - started
        )

        try:

            data = response.json()

            content = (
                data[
                    "choices"
                ][
                    0
                ][
                    "message"
                ].get(
                    "content",
                    "",
                )
            )

        except Exception as exc:

            raise ValueError(
                "NVIDIA NIM returned an unexpected "
                f"response format: {exc}"
            ) from exc

        _flush_print(
            f"[NIM] Response received in "
            f"{elapsed:.2f}s"
        )

        return (
            data,
            content,
            elapsed,
        )

    raise last_exception or RuntimeError(
        "Kimi K3 upstream request failed."
    )


def call_k3(
    images,
    invoice_filename,
    chart_of_accounts,
):

    if not images:
        raise ValueError(
            "At least one invoice page is required."
        )

    coa = _normalize_zoho_coa(
        chart_of_accounts
    )

    if not coa:
        raise ValueError(
            "Backend must supply the complete active Zoho COA."
        )

    acquired = NIM_REQUEST_LOCK.acquire(
        timeout=5
    )

    if not acquired:
        raise K3RateLimitError(
            "Another Kimi K3 invoice request is already being processed."
        )

    try:

        payload = _build_nim_payload(
            images,
            invoice_filename,
            coa,
        )

        (
            response_data,
            content,
            elapsed,
        ) = _post_to_nim(
            payload
        )

        prediction = _extract_json_object(
            content
        )

        # Lightweight structural check only.
        # There is deliberately NO JSON Schema validation.
        _basic_fixed_structure_check(
            prediction
        )

        # Runtime COA safety check.
        _validate_coa_runtime_ids(
            prediction,
            coa,
        )

        # ========================================================
        # LIVE EXTRACTION LOG
        # ========================================================

        print()
        print("=" * 110)
        print(
            "✅ KIMI K3 EXTRACTION SUCCESS",
            flush=True,
        )
        print("=" * 110)
        print(
            f"Invoice : {invoice_filename}",
            flush=True,
        )
        print(
            f"Model   : {MODEL}",
            flush=True,
        )
        print(
            f"Pages   : {len(images)}",
            flush=True,
        )
        print(
            f"Latency : {elapsed:.2f} sec",
            flush=True,
        )
        print("-" * 110)
        print(
            "📄 EXTRACTED DATA",
            flush=True,
        )
        print("-" * 110)
        print(
            json.dumps(
                prediction,
                indent=2,
                ensure_ascii=False,
            ),
            flush=True,
        )
        print("=" * 110)
        print(flush=True)

        return {
            "prediction": prediction,
            "usage": response_data.get(
                "usage",
                {},
            ),
            "elapsed_seconds": round(
                elapsed,
                2,
            ),
            "pages_sent": len(images),
            "zoho_coa_accounts_supplied": len(coa),
        }

    finally:
        NIM_REQUEST_LOCK.release()


print("NVIDIA NIM Kimi K3 client ready.")
print("Fixed JSON structure: ON")
print("JSON Schema lock: OFF")
print("Live extraction logs: ON")


In [ ]:
# 8) FastAPI application — live request logging

class K3InvoiceRequest(BaseModel):
    image_base64: str
    filename: Optional[str] = None

    chart_of_accounts: List[
        Dict[str, Any]
    ] = Field(
        default_factory=list
    )


k3_app = FastAPI(
    title="Kimi K3 Production Invoice Intelligence API",
    version="fixed-json",
)


def _decode_base64_payload(
    raw_b64: str,
):
    if not raw_b64:
        raise HTTPException(
            status_code=400,
            detail="image_base64 is empty.",
        )

    payload = (
        raw_b64.split(",", 1)[1]
        if (
            raw_b64.startswith("data:")
            and "," in raw_b64
        )
        else raw_b64
    )

    try:
        decoded = base64.b64decode(
            payload,
            validate=True,
        )
    except Exception as exc:
        raise HTTPException(
            status_code=400,
            detail=f"Invalid base64 invoice payload: {exc}",
        )

    if not decoded:
        raise HTTPException(
            status_code=400,
            detail="Decoded invoice payload is empty.",
        )

    return decoded


async def _process_invoice_request(
    req: K3InvoiceRequest,
):
    request_id = (
        "K3-"
        + datetime.datetime.now().strftime(
            "%Y%m%d-%H%M%S-%f"
        )
    )

    started = time.time()

    filename = (
        req.filename
        or "invoice.pdf"
    )

    print()
    print("#" * 110)
    print(
        "📥 BACKEND REQUEST RECEIVED",
        flush=True,
    )
    print("#" * 110)
    print(
        f"Request ID : {request_id}",
        flush=True,
    )
    print(
        f"Filename   : {filename}",
        flush=True,
    )
    print(
        f"COA Count  : "
        f"{len(req.chart_of_accounts or [])}",
        flush=True,
    )
    print("#" * 110)
    print(flush=True)

    if not req.chart_of_accounts:
        raise HTTPException(
            status_code=400,
            detail=(
                "chart_of_accounts is required. "
                "Send the complete active Zoho COA."
            ),
        )

    raw_bytes = _decode_base64_payload(
        req.image_base64
    )

    try:

        input_type, pages = (
            decode_invoice_to_pages(
                raw_bytes,
                filename,
            )
        )

    except HTTPException:
        raise

    except Exception as exc:

        print(
            f"[{request_id}] ❌ "
            f"Invoice decode error: {exc}",
            flush=True,
        )

        raise HTTPException(
            status_code=400,
            detail=str(exc),
        )

    print(
        f"[{request_id}] Input type : {input_type}",
        flush=True,
    )

    print(
        f"[{request_id}] Pages      : {len(pages)}",
        flush=True,
    )

    print(
        f"[{request_id}] 🚀 "
        f"Sending to {MODEL}...",
        flush=True,
    )

    try:

        # Keep the API server responsive while the
        # synchronous NIM call runs.
        result = await __import__(
            "asyncio"
        ).to_thread(
            call_k3,
            pages,
            filename,
            req.chart_of_accounts,
        )

    except K3RateLimitError as exc:

        print(
            f"[{request_id}] ❌ NIM RATE LIMIT: {exc}",
            flush=True,
        )

        raise HTTPException(
            status_code=429,
            detail=str(exc),
            headers={
                "Retry-After": "15"
            },
        )

    except TimeoutError as exc:

        print(
            f"[{request_id}] ❌ NIM TIMEOUT: {exc}",
            flush=True,
        )

        raise HTTPException(
            status_code=504,
            detail=str(exc),
        )

    except HTTPException:
        raise

    except Exception as exc:

        print(
            f"[{request_id}] ❌ "
            f"Runtime error: "
            f"{type(exc).__name__}: {exc}",
            flush=True,
        )

        traceback.print_exc()

        raise HTTPException(
            status_code=500,
            detail=(
                f"Kimi K3 inference failed: "
                f"{type(exc).__name__}: {exc}"
            ),
        )

    total_latency = (
        time.time()
        - started
    )

    print()
    print("#" * 110)
    print(
        "📤 RESPONSE SENT TO BACKEND",
        flush=True,
    )
    print("#" * 110)
    print(
        f"Request ID : {request_id}",
        flush=True,
    )
    print(
        f"Filename   : {filename}",
        flush=True,
    )
    print(
        f"Total Time : {total_latency:.2f} sec",
        flush=True,
    )
    print(
        f"Pages      : {len(pages)}",
        flush=True,
    )
    print(
        f"COA Count  : "
        f"{result.get('zoho_coa_accounts_supplied', 0)}",
        flush=True,
    )
    print("#" * 110)
    print(flush=True)

    return {
        "request_id": request_id,
        "status": "success",
        "model": MODEL,
        "input_type": input_type,
        "pages_processed": len(pages),
        "latency_seconds": round(
            total_latency,
            2,
        ),
        "prediction": result[
            "prediction"
        ],
        "usage": result.get(
            "usage",
            {},
        ),
        "zoho_coa_accounts_supplied":
            result.get(
                "zoho_coa_accounts_supplied",
                0,
            ),
    }


@k3_app.get("/health")
async def health():
    return {
        "status": "ok",
        "service": "kimi-k3-invoice-intelligence",
        "model": MODEL,
        "output_format": "fixed_json_structure",
        "schema_lock": False,
        "knowledge_base": KB_PATH.name,
        "knowledge_characters":
            len(KNOWLEDGE_BASE_TEXT),
        "runtime_coa":
            "complete_backend_supplied_zoho_coa",
    }


@k3_app.post(
    "/api/infer/k3-invoice"
)
async def infer_k3_invoice(
    req: K3InvoiceRequest,
):
    return await _process_invoice_request(
        req
    )


@k3_app.post(
    "/api/infer/extract-invoice"
)
async def extract_invoice_alias(
    req: K3InvoiceRequest,
):
    return await _process_invoice_request(
        req
    )


print(
    "FastAPI application ready."
)


## Backend request contract

Send the invoice file and the complete active Zoho COA.

```json
{
  "image_base64": "<BASE64_INVOICE_FILE>",
  "filename": "invoice.pdf",
  "chart_of_accounts": [
    {
      "account_id": "ZOHO-ID-1",
      "account_name": "Professional Fees",
      "account_type": "Expense",
      "description": "Professional and consulting service expenses"
    }
  ]
}
```

Production requests should include the **full active Zoho COA**, not a subset.


In [ ]:
# 9) Foreground FastAPI server + live ngrok
#
# IMPORTANT:
# This cell intentionally stays RUNNING for the duration of the demo.
# Requests, NIM logs and extracted JSON appear live in this cell output.
#
# Do NOT interrupt this cell while the backend is using the API.

K3_PORT = 8000

# Resolve ngrok token.
NGROK_AUTH_TOKEN = None

try:
    from google.colab import userdata
    NGROK_AUTH_TOKEN = userdata.get(
        "NGROK_AUTH_TOKEN"
    )
except Exception:
    pass

if not NGROK_AUTH_TOKEN:
    NGROK_AUTH_TOKEN = getpass(
        "Enter ngrok auth token: "
    ).strip()

if not NGROK_AUTH_TOKEN:
    raise ValueError(
        "NGROK_AUTH_TOKEN is required."
    )

# Clean up previous tunnel if this cell had been run before.
try:
    ngrok.kill()
except Exception:
    pass

ngrok.set_auth_token(
    NGROK_AUTH_TOKEN
)

# Open tunnel BEFORE foreground server starts.
# ngrok can wait for the local service to become available.
tunnel = ngrok.connect(
    K3_PORT,
    "http",
)

PUBLIC_URL = tunnel.public_url

config = uvicorn.Config(
    k3_app,
    host="0.0.0.0",
    port=K3_PORT,
    log_level="info",
    access_log=True,
)

server = uvicorn.Server(
    config
)

server.install_signal_handlers = (
    lambda: None
)

print()
print("=" * 110)
print(
    "🚀 KIMI K3 LIVE SERVER STARTING"
)
print("=" * 110)
print(
    "Model      :",
    MODEL,
)
print(
    "Port       :",
    K3_PORT,
)
print(
    "KB         :",
    KB_PATH.name,
)
print(
    "JSON       : FIXED STRUCTURE",
)
print(
    "Schema lock: OFF",
)
print(
    "Server     : FOREGROUND / CONTINUOUS",
)
print("=" * 110)
print(
    "PUBLIC URL :",
    PUBLIC_URL,
)
print(
    "HEALTH     :",
    f"{PUBLIC_URL}/health",
)
print(
    "K3 API     :",
    f"{PUBLIC_URL}/api/infer/k3-invoice",
)
print(
    "ALIAS      :",
    f"{PUBLIC_URL}/api/infer/extract-invoice",
)
print("=" * 110)
print(
    "✅ BACKEND CAN NOW SEND REQUESTS"
)
print(
    "✅ LIVE REQUEST + EXTRACTION LOGS WILL APPEAR BELOW"
)
print(
    "⚠️ KEEP THIS CELL RUNNING DURING THE DEMO"
)
print("=" * 110)
print(
    "Waiting for backend requests..."
)
print()

# ------------------------------------------------------------
# FOREGROUND SERVER
# ------------------------------------------------------------
# This call intentionally blocks the notebook cell.
# That is what keeps the service live and shows logs here.

server.run()


In [ ]:
# 10) Optional clean stop helper
#
# Run manually ONLY after the demo.

def stop_k3_service():
    global server

    try:
        if "server" in globals() and server:
            server.should_exit = True
    except Exception:
        pass

    time.sleep(1)

    try:
        ngrok.kill()
    except Exception:
        pass

    print(
        "Kimi K3 FastAPI + ngrok service stopped."
    )
